In [1]:
import torch
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from collections import Counter
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cpu")
MIXED_PRECISION = False

DATA_PATH = "/content/drive/MyDrive/blogtext.csv"

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))

def bucket_age(a):
    try:
        a = int(a)
    except:
        return "unknown"
    if 13 <= a <= 17: return "10s"
    if 23 <= a <= 27: return "20s"
    if 33 <= a <= 47: return "30s"
    return "unknown"

df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str)
df["gender"] = df["gender"].astype(str).str.lower().map(
    {"male":"male","m":"male","female":"female","f":"female"}
).fillna("unknown")
df["age_bucket"] = df["age"].apply(bucket_age).astype(str)
df["industry"] = df["topic"].astype(str)

df = df.sample(10_000, random_state=42).reset_index(drop=True)

print("\nLabel counts:")
print(" gender:", Counter(df["gender"]))
print(" age_bucket:", Counter(df["age_bucket"]))
ind_counts = Counter(df["industry"])
print(" industries (top 15):", ind_counts.most_common(15))

CLEAN_PATH = "/content/drive/MyDrive/mtap_clean_10k.csv"
df[["text","gender","age_bucket","industry"]].to_csv(CLEAN_PATH, index=False)
print("\nSaved cleaned subset to:", CLEAN_PATH)

needed_cols = ["text", "gender", "age_bucket", "industry"]
for col in needed_cols:
    assert col in df.columns, f"Missing column: {col}"

df_final = df[needed_cols].copy()
print("Final dataframe shape:", df_final.shape)
print(df_final.head(3))

# Save
CLEAN_PATH = "/content/drive/MyDrive/mtap_clean_10k.csv"
df_final.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned dataset to:", CLEAN_PATH)

df = df_final if 'df_final' in globals() else pd.read_csv(CLEAN_PATH)

# Basic guard
df['stratum'] = df['gender'].astype(str) + '|' + df['age_bucket'].astype(str)
MIN_PER_STRAT = 8  # ensures ~20% tmp has ≥2 → can split 1/1 into val/test
counts = df['stratum'].value_counts()
kept_strata = counts[counts >= MIN_PER_STRAT].index
dropped_strata = sorted(set(counts.index) - set(kept_strata))
df = df[df['stratum'].isin(kept_strata)].reset_index(drop=True)

df = df[df['text'].astype(str).str.strip().str.len() > 0].reset_index(drop=True)

train_df, tmp_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['stratum']
)
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, random_state=42, stratify=tmp_df['stratum']
)

for frame in (train_df, val_df, test_df):
    frame.drop(columns=['stratum'], inplace=True)

def distro(frame):
    return {
        "n": len(frame),
        "gender": Counter(frame["gender"]).most_common(),
        "age_bucket": Counter(frame["age_bucket"]).most_common(),
        "industry_top10": Counter(frame["industry"]).most_common(10),
    }

print("Dropped strata (too few samples):", dropped_strata)
print("train/val/test sizes:", len(train_df), len(val_df), len(test_df))

# Save splits
TRAIN_PATH = "/content/drive/MyDrive/mtap_train.csv"
VAL_PATH   = "/content/drive/MyDrive/mtap_val.csv"
TEST_PATH  = "/content/drive/MyDrive/mtap_test.csv"
train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
print("\nSaved:", TRAIN_PATH, VAL_PATH, TEST_PATH)

# Load splits
TRAIN_PATH = "/content/drive/MyDrive/mtap_train.csv"
VAL_PATH   = "/content/drive/MyDrive/mtap_val.csv"
TEST_PATH  = "/content/drive/MyDrive/mtap_test.csv"
train_df = train_df if 'train_df' in globals() else pd.read_csv(TRAIN_PATH)
val_df   = val_df   if 'val_df'   in globals() else pd.read_csv(VAL_PATH)
test_df  = test_df  if 'test_df'  in globals() else pd.read_csv(TEST_PATH)

Mounted at /content/drive
Rows: 681284
Columns: ['id', 'gender', 'age', 'topic', 'sign', 'date', 'text']

Label counts:
 gender: Counter({'male': 5047, 'female': 4953})
 age_bucket: Counter({'20s': 4688, '10s': 3438, '30s': 1818, 'unknown': 56})
 industries (top 15): [('indUnk', 3682), ('Student', 2335), ('Technology', 564), ('Education', 441), ('Arts', 439), ('Communications-Media', 311), ('Internet', 243), ('Non-Profit', 196), ('Engineering', 180), ('Law', 139), ('Publishing', 106), ('Science', 100), ('Government', 100), ('BusinessServices', 87), ('Religion', 84)]

Saved cleaned subset to: /content/drive/MyDrive/mtap_clean_10k.csv
Final dataframe shape: (10000, 4)
                                                text  gender age_bucket  \
0          1 Corinthians 5:11 'But now I have wri...  female        20s   
1         Having moved to Jersey City nearly a mo...  female        20s   
2                urlLink Hey, Metro! Need To Save...  female        20s   

  industry  
0   indUnk 

In [ ]:
# data preparation and training pipeline for LDA
SEED       = 42
NUM_TOPICS = 30
LDA_ITER   = 100

vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents="ascii",
    token_pattern=r"\b\w+\b",
    min_df=5, max_df=0.5, max_features=50000
)
X_tr = vectorizer.fit_transform(train_df["text"])
X_va = vectorizer.transform(val_df["text"])
X_te = vectorizer.transform(test_df["text"])

# LDA (variational Bayes)
lda = LatentDirichletAllocation(
    n_components=NUM_TOPICS,
    learning_method="batch",
    max_iter=LDA_ITER,
    random_state=SEED,
    evaluate_every=-1
)
lda.fit(X_tr)

# θ = doc-topic distributions
Theta_train = lda.transform(X_tr).astype(np.float32)
Theta_val   = lda.transform(X_va).astype(np.float32)
Theta_test  = lda.transform(X_te).astype(np.float32)

# Sanity checks
assert Theta_train.shape[0] == len(train_df)
assert Theta_val.shape[0]   == len(val_df)
assert Theta_test.shape[0]  == len(test_df)
print("θ shapes:", Theta_train.shape, Theta_val.shape, Theta_test.shape)
print("Row sum check (first row):", float(Theta_train[0].sum()))

# Save artifacts
VECT_PATH = "/content/drive/MyDrive/mtap_cv.pkl"
LDA_PATH  = "/content/drive/MyDrive/mtap_sklearn_lda.pkl"
NPY_TRAIN = "/content/drive/MyDrive/mtap_theta_train.npy"
NPY_VAL   = "/content/drive/MyDrive/mtap_theta_val.npy"
NPY_TEST  = "/content/drive/MyDrive/mtap_theta_test.npy"

joblib.dump(vectorizer, VECT_PATH)
joblib.dump(lda,        LDA_PATH)
np.save(NPY_TRAIN, Theta_train); np.save(NPY_VAL, Theta_val); np.save(NPY_TEST, Theta_test)

print("Saved:", VECT_PATH, "|", LDA_PATH)
print("Saved:", NPY_TRAIN, NPY_VAL, NPY_TEST)


θ shapes: (7961, 30) (995, 30) (996, 30)
Row sum check (first row): 1.0000001192092896
Saved: /content/drive/MyDrive/mtap_cv.pkl | /content/drive/MyDrive/mtap_sklearn_lda.pkl
Saved: /content/drive/MyDrive/mtap_theta_train.npy /content/drive/MyDrive/mtap_theta_val.npy /content/drive/MyDrive/mtap_theta_test.npy


In [ ]:
import re, json
import numpy as np, pandas as pd
from collections import Counter
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
Theta_train = np.load(NPY_TRAIN)
Theta_val   = np.load(NPY_VAL)
Theta_test  = np.load(NPY_TEST)

assert len(train_df)==len(Theta_train) and len(val_df)==len(Theta_val) and len(test_df)==len(Theta_test)

# Tokenizers & encoders
def word_tokens(s): return re.findall(r"\b\w+\b", str(s).lower())

# Char vocab
CHARSET = list("abcdefghijklmnopqrstuvwxyz0123456789 .,;:!?()[]{}'\"-_/\\@#*$%&+|")
PAD_CHAR = "<pad>"
CHARSET = [PAD_CHAR] + CHARSET
CHAR2IDX = {c:i for i,c in enumerate(CHARSET)}

# Hyperparams (fallbacks if not set)
MAX_CHAR_LEN = 1000
MAX_TOKENS   = 250
BATCH_SIZE   = 16

# for characters
def text_to_char_ids(text, max_len=MAX_CHAR_LEN):
    ids = [CHAR2IDX.get(c, 0) for c in str(text).lower()][:max_len]
    if len(ids) < max_len: ids += [0]*(max_len-len(ids))
    return np.asarray(ids, dtype=np.int64)

# for words
min_freq = 2
wc = Counter(t for toks in map(word_tokens, train_df["text"]) for t in toks)
vocab = ["<pad>","<unk>"] + [w for w,c in wc.items() if c>=min_freq]
W2I = {w:i for i,w in enumerate(vocab)}
def toks_to_word_ids(toks, max_len=MAX_TOKENS):
    ids = [W2I.get(w, 1) for w in toks[:max_len]]  # 1=<unk>
    if len(ids) < max_len: ids += [0]*(max_len-len(ids))  # 0=<pad>
    return np.asarray(ids, dtype=np.int64)

# Label spaces
GENDER_SET = sorted(train_df["gender"].astype(str).unique())
AGE_SET    = sorted(train_df["age_bucket"].astype(str).unique())
IND_SET    = sorted(train_df["industry"].astype(str).unique())
if "other" not in IND_SET: IND_SET = IND_SET + ["other"]

G2I = {g:i for i,g in enumerate(GENDER_SET)}
A2I = {a:i for i,a in enumerate(AGE_SET)}
I2I = {x:i for i,x in enumerate(IND_SET)}

def map_ind(label):
    return I2I[label] if label in I2I else I2I["other"]

# Dataset
class MTAPSet(Dataset):
    def __init__(self, df, theta):
        self.texts = df["text"].astype(str).tolist()
        self.g     = [G2I[str(x)] for x in df["gender"].astype(str)]
        self.a     = [A2I[str(x)] for x in df["age_bucket"].astype(str)]
        self.i     = [map_ind(str(x)) for x in df["industry"].astype(str)]
        self.theta = theta.astype(np.float32)
        self.toks  = [word_tokens(t) for t in self.texts]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return {
            "char_ids": torch.from_numpy(text_to_char_ids(self.texts[idx])),
            "word_ids": torch.from_numpy(toks_to_word_ids(self.toks[idx])),
            "theta":    torch.from_numpy(self.theta[idx]),
            "y_gender": torch.tensor(self.g[idx], dtype=torch.long),
            "y_age":    torch.tensor(self.a[idx], dtype=torch.long),
            "y_ind":    torch.tensor(self.i[idx], dtype=torch.long),
        }

train_ds = MTAPSet(train_df, Theta_train)
val_ds   = MTAPSet(val_df,   Theta_val)
test_ds  = MTAPSet(test_df,  Theta_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

# Save mappings for reproducibility
MAP_PATH = "/content/drive/MyDrive/mtap_label_and_vocab.json"
with open(MAP_PATH, "w") as f:
    json.dump({
        "GENDER_SET": GENDER_SET,
        "AGE_SET": AGE_SET,
        "IND_SET": IND_SET,
        "vocab_size": len(vocab),
        "NUM_TOPICS": int(Theta_train.shape[1])
    }, f, indent=2)

# Quick sanity check
print("Datasets:", len(train_ds), len(val_ds), len(test_ds))
batch = next(iter(train_loader))
print("Batch shapes:",
      batch["char_ids"].shape, batch["word_ids"].shape, batch["theta"].shape)
print("Saved mapping:", MAP_PATH)


Datasets: 7961 995 996
Batch shapes: torch.Size([16, 1000]) torch.Size([16, 250]) torch.Size([16, 30])
Saved mapping: /content/drive/MyDrive/mtap_label_and_vocab.json


In [ ]:
import numpy as np, torch, torch.nn as nn
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from IPython.display import clear_output
from copy import deepcopy

device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
MIXED_PRECISION = False if device.type == "cpu" else True

SEED        = 42
BATCH_SIZE  = 32

# Optimizer
OPTIM       = "sgd"
LR          = 2e-3
WEIGHT_DECAY= 5e-4
MOMENTUM    = 0.9

MAX_CHAR_LEN = 4000         # chars per doc
MAX_TOKENS   = 256          # words per doc

# LDA
NUM_TOPICS   = 30
LDA_ITER     = 100

# Training
EPOCHS       = 50
ES_PATIENCE  = 50
ES_MIN_DELTA = 1e-4

# MTAP dims
T             = 256
CHAR_EMB_DIM  = 32
CHAR_HIDDEN   = 128
WORD_EMB_DIM  = 100
KERNEL_SIZES  = (3,4,5)
CHANNELS_PER_WIDTH = 32
WORD_CNN_TOTAL     = len(KERNEL_SIZES) * CHANNELS_PER_WIDTH
TOPIC_OUT     = T    # project θ -> T

# NOTE: Paper fusion = Hadamard (element-wise) product, so no extra HIDDEN_FC needed.
NUM_GENDER = len(GENDER_SET); NUM_AGE = len(AGE_SET); NUM_IND = len(IND_SET)
NUM_TOPICS = Theta_train.shape[1]; VOCAB_SIZE = len(W2I); CHAR_VOCAB = len(CHARSET)

class CharBiLSTMDoc(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, T, pad_idx=0, dropout=0.20):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(2 * hidden, T)
        self.act  = nn.ReLU()

    def forward(self, x):              # x: [B, L_char]
        e, _ = self.emb(x), None       # [B, L_char, emb_dim]
        e = self.drop(e)
        h, _ = self.lstm(e)            # [B, L_char, 2*hidden]
        h = h.max(dim=1).values        # max-over-time pool → [B, 2*hidden]
        z = self.act(self.proj(h))     # [B, T]
        return z


class WordCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=100, channels_per=32, kernel_sizes=(3,4,5),
                 T=256, pad_idx=0, dropout=0.20):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        # one 1D conv per n-gram width; outputs 'channels_per' maps each
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, channels_per, k, padding=k//2) for k in kernel_sizes
        ])
        self.act  = nn.ReLU()
        # concat all conv features, then project to the common dim T
        self.proj = nn.Linear(channels_per * len(kernel_sizes), T)

    def forward(self, x):                    # x: [B, L_words]
        e = self.drop(self.emb(x)).transpose(1, 2)   # [B, E, L]
        feats = []
        for conv in self.convs:
            h = self.act(conv(e))                    # [B, C, L]
            h = torch.max(h, dim=-1).values          # global max-pool → [B, C]
            feats.append(h)
        h = torch.cat(feats, dim=1)                  # [B, C * #kernels]
        z = self.act(self.proj(h))                   # [B, T]  ← word path output
        return z


class TopicMLP(nn.Module):
    def __init__(self, in_dim: int, T: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, T),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)  # [B, T]


class MTAP(nn.Module):
    """
    Paper-aligned MTAP:
      - Char path: BiLSTM over characters → max-pool → Linear → ReLU → [B, T]
      - Word path: CNN over words (k in {3,4,5}) + max-over-time → Linear → ReLU → [B, T]
      - Topic path: LDA θ → Linear → ReLU → [B, T]
      - Fusion: Hadamard (element-wise) product of the three T-dim vectors
      - Heads: Linear(T → #classes) for gender, age, job/industry
      - Loss (outside this class): sum of 3 cross-entropies
    """
    def __init__(self):
        super().__init__()
        # encoders
        self.char = CharBiLSTMDoc(
            vocab_size=CHAR_VOCAB,
            emb_dim=CHAR_EMB_DIM,
            hidden=CHAR_HIDDEN,
            T=T,
            pad_idx=0,
            dropout=0.20,
        )
        self.word = WordCNN(
            vocab_size=VOCAB_SIZE,
            emb_dim=WORD_EMB_DIM,
            channels_per=CHANNELS_PER_WIDTH,
            kernel_sizes=KERNEL_SIZES,
            T=T,
            pad_idx=0,
            dropout=0.20,
        )
        self.topi = TopicMLP(in_dim=NUM_TOPICS, T=T, dropout=0.0)

        # classification heads (no extra MLP; directly from fused T)
        self.head_g = nn.Linear(T, NUM_GENDER)
        self.head_a = nn.Linear(T, NUM_AGE)
        self.head_i = nn.Linear(T, NUM_IND)

    def forward(self, batch):
        zc = self.char(batch["char_ids"])   # [B, T]
        zw = self.word(batch["word_ids"])   # [B, T]
        zt = self.topi(batch["theta"])      # [B, T]

        # Hadamard fusion (paper)
        h = zc * zw * zt                    # [B, T]

        # logits for each task
        lg = self.head_g(h)                 # [B, NUM_GENDER]
        la = self.head_a(h)                 # [B, NUM_AGE]
        li = self.head_i(h)                 # [B, NUM_IND]
        return lg, la, li

model = MTAP().to(device)

CEg = nn.CrossEntropyLoss()
CEa = nn.CrossEntropyLoss()
CEi = nn.CrossEntropyLoss()

opt = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=False
)

scaler = None

def run_epoch(loader, model, opt, CEg, CEa, CEi, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    tot_loss = 0.0
    gT, gP, aT, aP, iT, iP = [], [], [], [], [], []

    for batch in loader:
        # to device
        for k in batch:
            batch[k] = batch[k].to(device)

        # forward
        with torch.set_grad_enabled(train):
            lg, la, li = model(batch)
            Lg = CEg(lg, batch["y_gender"])
            La = CEa(la, batch["y_age"])
            Li = CEi(li, batch["y_ind"])
            loss = Lg + La + Li   # ← paper: sum of 3 CE losses

            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

        bs = batch["char_ids"].size(0)
        tot_loss += loss.item() * bs

        gT += batch["y_gender"].tolist(); gP += lg.argmax(1).tolist()
        aT += batch["y_age"].tolist();    aP += la.argmax(1).tolist()
        iT += batch["y_ind"].tolist();    iP += li.argmax(1).tolist()

    n = len(loader.dataset)
    import numpy as np
    return {
        "loss": tot_loss / n,
        "g_acc": (np.array(gT) == np.array(gP)).mean(),
        "a_acc": (np.array(aT) == np.array(aP)).mean(),
        "i_acc": (np.array(iT) == np.array(iP)).mean(),
    }

# training loop
import matplotlib.pyplot as plt
from IPython.display import clear_output
from copy import deepcopy

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss = 0.0
    gT,gP,aT,aP,iT,iP = [],[],[],[],[],[]

    for batch in loader:
        for k in batch: batch[k] = batch[k].to(device)
        if train: opt.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            lg, la, li = model(batch)
            Lg = CEg(lg, batch["y_gender"])
            La = CEa(la, batch["y_age"])
            Li = CEi(li, batch["y_ind"])
            loss = Lg + La + Li

        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        bs = batch["char_ids"].size(0)
        tot_loss += loss.item() * bs

        gT += batch["y_gender"].tolist(); gP += lg.argmax(1).tolist()
        aT += batch["y_age"].tolist();    aP += la.argmax(1).tolist()
        iT += batch["y_ind"].tolist();    iP += li.argmax(1).tolist()

    n = len(loader.dataset)
    g_acc = (np.array(gT) == np.array(gP)).mean()
    a_acc = (np.array(aT) == np.array(aP)).mean()
    i_acc = (np.array(iT) == np.array(iP)).mean()
    return {"loss": tot_loss / n, "g_acc": g_acc, "a_acc": a_acc, "i_acc": i_acc}

# Train -early stopping on val loss
history = {"train": {"loss": []}, "val": {"loss": []}}
best_state, best_val_loss = None, float("inf")
patience = 0
epochs_ran = 0

for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    epochs_ran = ep

    history["train"]["loss"].append(tr["loss"])
    history["val"]["loss"].append(va["loss"])

    clear_output(wait=True)
    print(f"Epoch {ep}/{EPOCHS}")
    print(f"Train Loss: {tr['loss']:.4f} | Val Loss: {va['loss']:.4f}")
    print(f"Train Acc (%): G/A/I = {tr['g_acc']*100:.1f}/{tr['a_acc']*100:.1f}/{tr['i_acc']*100:.1f}")
    print(f"Val   Acc (%): G/A/I = {va['g_acc']*100:.1f}/{va['a_acc']*100:.1f}/{va['i_acc']*100:.1f}")

    # Early stopping (minimize val loss)
    if best_val_loss - va["loss"] > ES_MIN_DELTA:
        best_val_loss = va["loss"]
        best_state = deepcopy(model.state_dict())
        patience = 0
    else:
        patience += 1
        if patience >= ES_PATIENCE:
            print(f"Early stop at epoch {ep} (best val loss = {best_val_loss:.4f})")
            break

# Load best weights and evaluate on TEST
if best_state is not None:
    model.load_state_dict(best_state)

te = run_epoch(test_loader, train=False)
print("\n=== TEST ===")
print({
    "test_loss": round(te["loss"], 4),
    "gender_acc%": round(te["g_acc"]*100, 1),
    "age_acc%":    round(te["a_acc"]*100, 1),
    "ind_acc%":    round(te["i_acc"]*100, 1),
})

# Single plot: Train loss per epoch + Test loss line
epochs = list(range(1, epochs_ran + 1))
plt.figure(figsize=(7,4))
plt.plot(epochs, history["train"]["loss"], label="Train loss")
plt.hlines(te["loss"], xmin=1, xmax=epochs_ran, linestyles="--",
           label=f"Test loss = {te['loss']:.4f}")
plt.xlabel("Epoch"); plt.ylabel("Total loss"); plt.title("MTAP: Train vs Test Loss")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()




Epoch 25/50
Train Loss: 3.6485 | Val Loss: 4.0215
Train Acc (%): G/A/I = 59.3/63.5/40.9
Val   Acc (%): G/A/I = 56.3/55.5/37.2


In [ ]:
# Results

# Epoch 3/50
# Train Loss: 4.0995 | Val Loss: 4.1415
# Train Acc (%): G/A/I = 50.7/46.7/36.9
# Val   Acc (%): G/A/I = 50.5/46.8/35.7

# Epoch 4/50
# Train Loss: 4.0863 | Val Loss: 4.1271
# Train Acc (%): G/A/I = 50.2/46.7/36.8
# Val   Acc (%): G/A/I = 50.5/46.8/35.7

# Epoch 5/50
# Train Loss: 4.0715 | Val Loss: 4.1334
# Train Acc (%): G/A/I = 50.8/47.0/36.9
# Val   Acc (%): G/A/I = 50.5/46.8/35.7

# Epoch 7/50
# Train Loss: 4.0452 | Val Loss: 4.0638
# Train Acc (%): G/A/I = 52.7/49.4/36.9
# Val   Acc (%): G/A/I = 55.4/46.8/35.7

# Epoch 8/50
# Train Loss: 4.0246 | Val Loss: 4.0474
# Train Acc (%): G/A/I = 55.7/51.8/37.0
# Val   Acc (%): G/A/I = 52.6/50.9/35.7

# Epoch 9/50
# Train Loss: 3.9918 | Val Loss: 4.0073
# Train Acc (%): G/A/I = 56.7/54.1/36.9
# Val   Acc (%): G/A/I = 56.2/53.9/35.8

# Epoch 10/50
# Train Loss: 3.9524 | Val Loss: 3.9735
# Train Acc (%): G/A/I = 57.6/55.0/37.6
# Val   Acc (%): G/A/I = 53.9/56.0/37.6

# Epoch 11/50
# Train Loss: 3.9383 | Val Loss: 4.0448
# Train Acc (%): G/A/I = 57.6/55.2/37.8
# Val   Acc (%): G/A/I = 52.5/52.8/35.7

# Epoch 12/50
# Train Loss: 3.9219 | Val Loss: 3.9704
# Train Acc (%): G/A/I = 57.8/55.4/37.6
# Val   Acc (%): G/A/I = 54.6/55.3/35.7

# Epoch 13/50
# Train Loss: 3.9127 | Val Loss: 3.9558
# Train Acc (%): G/A/I = 58.2/55.4/37.5
# Val   Acc (%): G/A/I = 55.7/56.7/35.7

# Epoch 14/50
# Train Loss: 3.8922 | Val Loss: 4.0168
# Train Acc (%): G/A/I = 58.6/56.0/37.5
# Val   Acc (%): G/A/I = 54.9/53.1/35.7

# Epoch 15/50
# Train Loss: 3.8830 | Val Loss: 3.9875
# Train Acc (%): G/A/I = 59.1/55.9/37.7
# Val   Acc (%): G/A/I = 57.1/54.3/35.8

# Epoch 16/50
# Train Loss: 3.8673 | Val Loss: 3.9411
# Train Acc (%): G/A/I = 59.0/56.6/37.9
# Val   Acc (%): G/A/I = 57.1/55.6/35.9

# Epoch 17/50
# Train Loss: 3.8421 | Val Loss: 3.9784
# Train Acc (%): G/A/I = 59.3/57.7/38.0
# Val   Acc (%): G/A/I = 54.5/55.8/36.0

# Epoch 18/50
# Train Loss: 3.8148 | Val Loss: 3.9545
# Train Acc (%): G/A/I = 59.5/58.4/38.3
# Val   Acc (%): G/A/I = 57.6/56.4/36.2

# Epoch 19/50
# Train Loss: 3.7879 | Val Loss: 3.9346
# Train Acc (%): G/A/I = 59.5/58.8/39.0
# Val   Acc (%): G/A/I = 57.9/57.3/36.6

# Epoch 20/50
# Train Loss: 3.7656 | Val Loss: 3.9397
# Train Acc (%): G/A/I = 59.8/60.1/38.9
# Val   Acc (%): G/A/I = 56.1/57.0/36.9

# Epoch 21/50
# Train Loss: 3.7383 | Val Loss: 3.9530
# Train Acc (%): G/A/I = 59.5/60.6/39.5
# Val   Acc (%): G/A/I = 56.7/56.6/36.7

# Epoch 22/50
# Train Loss: 3.7160 | Val Loss: 3.9887
# Train Acc (%): G/A/I = 59.6/61.6/39.7
# Val   Acc (%): G/A/I = 58.1/56.1/36.9

# Epoch 24/50
# Train Loss: 3.6713 | Val Loss: 3.9427
# Train Acc (%): G/A/I = 59.8/62.5/40.5
# Val   Acc (%): G/A/I = 58.2/57.4/37.0

# Epoch 25/50
# Train Loss: 3.6485 | Val Loss: 4.0215
# Train Acc (%): G/A/I = 59.3/63.5/40.9
# Val   Acc (%): G/A/I = 56.3/55.5/37.2